# Part F: Hugging Face Fine-Tuning and Transformer Explanation

This notebook uses 1,000 training records, 250 validation records, 64 tokens per sequence, and one CPU-only training epoch. These values are within the assessment caps of 4,000 training records, 1,000 validation records, 128 tokens, and two epochs. It uses BERT-tiny, a lightweight checkpoint selected for practical CPU execution.

In [33]:
import json
import sys
from pathlib import Path

import numpy as np
import torch
from datasets import load_dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from torch.optim import AdamW
from torch.utils.data import DataLoader
from transformers import AutoModelForSequenceClassification, AutoTokenizer

project_root = Path.cwd().resolve()
if not (project_root / 'src').is_dir():
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

np.random.seed(42)
torch.manual_seed(42)
device = torch.device('cpu')
checkpoint_name = 'prajjwal1/bert-tiny'
max_seq_length = 64
num_epochs = 1
train_batch_size = 16
validation_batch_size = 32
max_train_records = 1000
max_validation_records = 250
torch.set_num_threads(min(4, torch.get_num_threads()))

## F1: CPU-Constrained Fine-Tuning
AG News is shuffled with seed 42 and limited to a 1,000/250 subset, below the stated assessment caps. The model always uses CPU; no CUDA calls or GPU assumptions are used.

In [34]:
raw_dataset = load_dataset('ag_news')
train_dataset = raw_dataset['train'].shuffle(seed=42).select(range(min(max_train_records, len(raw_dataset['train']))))
validation_dataset = raw_dataset['test'].shuffle(seed=42).select(range(min(max_validation_records, len(raw_dataset['test']))))
label_names = raw_dataset['train'].features['label'].names
num_labels = len(label_names)

This output confirms the capped split sizes and the label order read directly from the AG News dataset metadata.

In [35]:
print(f'Training records: {len(train_dataset)}')
print(f'Validation records: {len(validation_dataset)}')
print(f'Labels: {label_names}')

Training records: 1000
Validation records: 250
Labels: ['World', 'Sports', 'Business', 'Sci/Tech']


The selected 1,000/250 samples are below the 4,000/1,000 assessment caps, and the dataset metadata supplies the four-class mapping rather than relying on an unverified hard-coded ordering.

The tokenizer uses fixed-length padding to 64 tokens. Fixed padding is chosen here for simple, predictable CPU batches; truncation keeps every input within the 128-token assessment limit.

In [36]:
tokenizer = AutoTokenizer.from_pretrained(checkpoint_name)

def tokenize_batch(batch):
    return tokenizer(batch['text'], truncation=True, max_length=max_seq_length, padding='max_length')

tokenized_train = train_dataset.map(tokenize_batch, batched=True, remove_columns=['text'])
tokenized_validation = validation_dataset.map(tokenize_batch, batched=True, remove_columns=['text'])
tokenized_train.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
tokenized_validation.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
train_loader = DataLoader(tokenized_train, batch_size=train_batch_size, shuffle=True)
validation_loader = DataLoader(tokenized_validation, batch_size=validation_batch_size)

c:\Users\Priya Koma\Desktop\AI_ML_Assessments\explainable-ml-assignment\venv\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


The model has four output labels matching AG News: World, Sports, Business, and Sci/Tech. BERT-tiny, the CPU device, one epoch, and the 1,000-record subset keep the manual loop practical while still demonstrating fine-tuning.

In [37]:
transformer_model = AutoModelForSequenceClassification.from_pretrained(checkpoint_name, num_labels=num_labels).to(device)
optimizer = AdamW(transformer_model.parameters(), lr=2e-5)
transformer_model.train()
for epoch in range(num_epochs):
    running_loss = 0.0
    for step, batch in enumerate(train_loader, start=1):
        optimizer.zero_grad()
        outputs = transformer_model(
            input_ids=batch['input_ids'].to(device),
            attention_mask=batch['attention_mask'].to(device),
            labels=batch['label'].to(device),
        )
        outputs.loss.backward()
        optimizer.step()
        running_loss += float(outputs.loss.item())
        if step % 10 == 0 or step == len(train_loader):
            print(f'Epoch {epoch + 1}/{num_epochs}, step {step}/{len(train_loader)}, mean loss: {running_loss / step:.4f}')

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at prajjwal1/bert-tiny and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch 1/1, step 10/63, mean loss: 1.3803
Epoch 1/1, step 20/63, mean loss: 1.3917
Epoch 1/1, step 30/63, mean loss: 1.3914
Epoch 1/1, step 40/63, mean loss: 1.3893
Epoch 1/1, step 50/63, mean loss: 1.3863
Epoch 1/1, step 60/63, mean loss: 1.3823
Epoch 1/1, step 63/63, mean loss: 1.3802


This output evaluates the fine-tuned model on the capped validation split using accuracy and macro-averaged precision, recall, and F1.

In [38]:
transformer_model.eval()
validation_predictions = []
validation_targets = []
with torch.no_grad():
    for batch in validation_loader:
        logits = transformer_model(input_ids=batch['input_ids'].to(device), attention_mask=batch['attention_mask'].to(device)).logits
        validation_predictions.extend(torch.argmax(logits, dim=1).cpu().tolist())
        validation_targets.extend(batch['label'].tolist())
macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(validation_targets, validation_predictions, average='macro', zero_division=0)
transformer_metrics = {
    'accuracy': float(accuracy_score(validation_targets, validation_predictions)),
    'macro_precision': float(macro_precision), 'macro_recall': float(macro_recall),
    'macro_f1': float(macro_f1), 'checkpoint': checkpoint_name,
    'num_train_records': int(len(train_dataset)), 'num_val_records': int(len(validation_dataset)),
    'max_seq_length': int(max_seq_length), 'num_epochs': int(num_epochs),
}
transformer_metrics

{'accuracy': 0.428,
 'macro_precision': 0.5474161255411255,
 'macro_recall': 0.4185568086883876,
 'macro_f1': 0.41836201079622126,
 'checkpoint': 'prajjwal1/bert-tiny',
 'num_train_records': 1000,
 'num_val_records': 250,
 'max_seq_length': 64,
 'num_epochs': 1}

Macro metrics give each of the four classes equal weight, making evaluation informative even if sampled label frequencies differ. All values are converted to JSON-serializable native types.

This inference demo applies the fine-tuned classifier to custom sentences spanning the four AG News topics.

In [39]:
custom_examples = [
    'Leaders met to discuss a new international climate agreement.',
    'The underdog team won the championship after a late goal.',
    'The company reported higher quarterly revenue and expanded its market.',
    'Researchers announced a new satellite launch and software platform.',
    'Investors reacted as global markets changed after an election.',
]
custom_tokens = tokenizer(custom_examples, truncation=True, max_length=max_seq_length, padding='max_length', return_tensors='pt')
with torch.no_grad():
    custom_probabilities = torch.softmax(transformer_model(input_ids=custom_tokens['input_ids'].to(device), attention_mask=custom_tokens['attention_mask'].to(device)).logits, dim=1)
for sentence, probabilities in zip(custom_examples, custom_probabilities):
    predicted_index = int(torch.argmax(probabilities).item())
    confidence = float(probabilities[predicted_index].item())
    print(f'{label_names[predicted_index]} ({confidence:.3f}): {sentence}')

Sci/Tech (0.260): Leaders met to discuss a new international climate agreement.
Sports (0.277): The underdog team won the championship after a late goal.
Business (0.277): The company reported higher quarterly revenue and expanded its market.
Business (0.284): Researchers announced a new satellite launch and software platform.
Business (0.293): Investors reacted as global markets changed after an election.


Each custom sentence is assigned one of the verified AG News labels with its softmax confidence. These examples demonstrate inference only; they are not drawn from the training or validation sample.

The next cell saves only the label mapping and evaluation metrics. The fine-tuned checkpoint is deliberately not saved to the repository; reproduce it by re-running this notebook. The README task should document that reproduction step, but this notebook task does not edit README.md.

In [40]:
artifact_directory = project_root / 'artifacts'
artifact_directory.mkdir(exist_ok=True)
label_mapping = {str(index): label for index, label in enumerate(label_names)}
with (artifact_directory / 'label_mapping.json').open('w', encoding='utf-8') as mapping_file:
    json.dump(label_mapping, mapping_file, indent=2)
with (artifact_directory / 'transformer_metrics.json').open('w', encoding='utf-8') as metrics_file:
    json.dump(transformer_metrics, metrics_file, indent=2)

The artifacts now contain the verified label mapping and compact metric record, while omitting model weights and checkpoint directories from the repository.

## F2: Transformer Explanation

**Tokenization.** The BERT-tiny tokenizer converts each AG News headline or article into token IDs, integers from its learned vocabulary. It uses subword tokenization, so frequent words may be one token while unfamiliar words can be split into meaningful pieces. In this notebook, truncation and fixed padding produce sequences of 64 IDs, allowing the CPU training batches to have a predictable shape.

**Input embeddings.** Token IDs have no numeric meaning by themselves. The model maps every ID to a dense learned vector, called an embedding, so similar contexts can be represented by related patterns of numbers. These vectors are the first continuous representation passed into the BERT-tiny encoder for the four-class World, Sports, Business, and Sci/Tech classifier.

**Positional information.** Self-attention alone has no built-in sense of first, next, or last token, unlike an RNN that processes a sequence in order. Transformer input therefore includes learned positional embeddings alongside token embeddings. This lets a sentence such as a sports headline retain word order, which can change its meaning.

**Self-attention.** For each token, the model creates conceptual query, key, and value representations. A query compares with keys from all tokens to assign attention weights, then uses those weights to combine their values. Thus a word like 'market' can attend differently to nearby words such as 'stock' or 'supermarket', making its representation context dependent.

**Transformer encoder.** BERT-tiny stacks compact encoder blocks, each combining multi-head self-attention with a feedforward network, residual connections, and normalization. Multiple attention heads can capture different relationships in parallel, while successive layers form increasingly abstract text features. The classification head added here consumes the final sequence representation and produces four logits.

**Pretraining.** BERT-tiny inherits BERT-family masked-language-model pretraining, where the model learns to infer masked tokens from surrounding context. That broad text objective exposes it to grammar, vocabulary, and semantic associations before this small AG News sample is seen. Pretraining therefore supplies useful general features that one CPU epoch could not learn from scratch.

**Fine-tuning.** Fine-tuning adds and trains a task-specific classification head while updating the pretrained model weights on labeled AG News examples. Here the labels direct the model toward the four news categories, and the one-epoch limit constrains compute while still demonstrating the adaptation workflow.

**Fine-tuning versus prompt engineering.** Fine-tuning changes model weights using labeled training data. Prompt engineering keeps weights frozen and changes behavior at inference time through instructions or examples in the input, so it needs neither weight updates nor a labeled training loop.

**Fine-tuning versus RAG.** Fine-tuning bakes learned behavior into parameters, whereas retrieval-augmented generation retrieves external documents at inference time and places them in context while leaving weights unchanged. RAG is particularly useful when facts change frequently or must be traceable to a source; this AG News classifier instead learns a stable label decision from its training sample.

(Word count: 542)